# SQL Practice - Day 4 (Advanced)

**15 Interview Questions (7–10 LPA)**

Topics:
- CTE
- ROW_NUMBER / RANK / DENSE_RANK
- GROUP BY / HAVING
- Date Functions
- Self Join
- Joins
- Correlated Subqueries
- Top-N Problems
- Aggregation on Aggregation

Each question creates `df` (and `df1` where required).

## Q1

**Task:** CTE: Find customers whose total spending is above average customer spending.

In [72]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'amount':[500,700,300,1000,200,800]})


In [74]:
sqldf("""
With my_cte as (
                select customer_id , sum(amount) as amount
                from df
                group by customer_id
                having sum(amount) > (select avg(sum_amount) 
                                        from ( select customer_id , sum(amount) as sum_amount
                                                from df 
                                                group by customer_id))
)
select * from  my_cte



""")

,customer_id,amount
0,101,2000


## Q2

**Task:** ROW_NUMBER(): Latest order for each customer.

In [11]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'order_date':['2025-01-01','2025-01-15','2025-01-05','2025-01-10','2025-01-20','2025-02-01'],'amount':[500,700,300,1000,200,800]})


In [17]:
sqldf(""" 
select *
from (select *,
        row_number()
        over(partition by customer_id  order by order_date desc) as rn
       from df) t
where rn = 1



""")

,order_id,customer_id,order_date,amount,rn
0,6,101,2025-02-01,800,1
1,5,102,2025-01-20,200,1
2,4,103,2025-01-10,1000,1


## Q3

**Task:** DENSE_RANK(): 2nd highest salary in each department.

In [18]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6],'name':['A','B','C','D','E','F'],'department':['HR','HR','HR','IT','IT','IT'],'salary':[50000,60000,70000,40000,45000,80000]})


In [20]:
sqldf("""
select *
from ( select * , 
        dense_rank()
        over(partition by department order by salary desc) as rn
        from df) t
where rn = 2 


""")

,emp_id,name,department,salary,rn
0,2,B,HR,60000,2
1,5,E,IT,45000,2


## Q4

**Task:** RANK(): Top 3 highest salaries in company.

In [22]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6,7],'name':['A','B','C','D','E','F','G'],'salary':[50000,60000,70000,70000,45000,80000,75000]})


In [27]:
sqldf("""
select *
from (select * ,
        rank()
        over(order by salary desc) as rn
        from df) t
where rn <= 3


""")

,emp_id,name,salary,rn
0,6,F,80000,1
1,7,G,75000,2
2,3,C,70000,3
3,4,D,70000,3


## Q5

**Task:** GROUP BY: First order date and total orders for each customer.

In [28]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'order_date':['2025-01-01','2025-01-15','2025-01-05','2025-01-10','2025-01-20','2025-02-01']})


In [30]:
sqldf("""
select customer_id , min(order_date) as first_order_date , count(order_id) as total_order
from df
group by customer_id



""")

,customer_id,first_order_date,total_order
0,101,2025-01-01,3
1,102,2025-01-05,2
2,103,2025-01-10,1


## Q6

**Task:** Date: Find total sales for each month.

In [31]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'order_date':['2025-01-01','2025-01-15','2025-02-05','2025-02-10','2025-03-20','2025-03-25'],'amount':[500,700,300,1000,200,800]})


,order_id,order_date,amount
0,1,2025-01-01,500
1,2,2025-01-15,700
2,3,2025-02-05,300
3,4,2025-02-10,1000
4,5,2025-03-20,200
5,6,2025-03-25,800


## Q7

**Task:** Self Join: Employees earning more than their manager.

In [35]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['A','B','C','D','E'],'manager_id':[None,1,1,2,2],'salary':[100000,60000,120000,70000,50000]})
df

,emp_id,name,manager_id,salary
0,1,A,NaN,100000
1,2,B,1.0,60000
2,3,C,1.0,120000
3,4,D,2.0,70000
4,5,E,2.0,50000


In [40]:
sqldf("""
select 
    e.name as emp_name,
    e.salary as emp_salary,
    m.name as manager_name,
    m.salary as manager_salary
from df e
join df m on e.manager_id = m.emp_id
where e.salary > m.salary


""")

,emp_name,emp_salary,manager_name,manager_salary
0,C,120000,A,100000
1,D,70000,B,60000


## Q8

**Task:** LEFT JOIN: Customers with no orders.

In [41]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104],'name':['Alice','Bob','Charlie','David']});df1=pd.DataFrame({'order_id':[1,2,3],'customer_id':[101,101,102]})


In [45]:
sqldf("""
select c.customer_id , o.order_id
from df c
left join df1 o on c.customer_id = o.customer_id
where o.order_id is null


""")

,customer_id,order_id
0,103,None
1,104,None


## Q9

**Task:** INNER JOIN: Customer name with total spending.

In [46]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103],'name':['Alice','Bob','Charlie']});df1=pd.DataFrame({'order_id':[1,2,3,4],'customer_id':[101,101,102,103],'amount':[500,700,300,1000]})


In [50]:
sqldf(""" 
select c.name , sum(o.amount) as total_spending
from df c
inner join df1 o  on c.customer_id = o.customer_id
group by c.customer_id


""")

,name,total_spending
0,Alice,1200
1,Bob,300
2,Charlie,1000


## Q10

**Task:** Correlated Subquery: Products priced above category average.

In [51]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'price':[100,150,200,250,300,350]})


In [52]:
sqldf("""
select product_id , category , price
from df p
where price > (select avg(price) 
                from df o
                where p.category = o.category)


""")

,product_id,category,price
0,3,A,200
1,6,B,350


## Q11

**Task:** Correlated Subquery: Employees below department average salary.

In [76]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E','F'],'department':['HR','HR','HR','IT','IT','IT'],'salary':[50000,60000,70000,40000,45000,80000]})


In [79]:
sqldf("""
select name , salary
from df a
where salary < (select avg(salary) 
                    from df b
                    where a.department = b.department)



""")

,name,salary
0,A,50000
1,D,40000
2,E,45000


## Q12

**Task:** Top-N: Top 2 customers by total spending.

In [82]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6,7],'customer_id':[101,101,102,103,102,101,104],'amount':[500,700,300,1000,200,800,1500]})
df

,order_id,customer_id,amount
0,1,101,500
1,2,101,700
2,3,102,300
3,4,103,1000
4,5,102,200
5,6,101,800
6,7,104,1500


In [88]:
sqldf(""" 
select customer_id , amount
from (select customer_id , sum(amount) as amount,
        dense_rank()
        over( order by sum(amount) desc ) as rn
        from df
        group by customer_id
)t
where rn <= 2
        



""")

,customer_id,amount
0,101,2000
1,104,1500


## Q13

**Task:** Aggregation on Aggregation: Department average salary greater than company average salary.

In [89]:
import pandas as pd
df=pd.DataFrame({'department':['HR','HR','HR','IT','IT','IT','Sales','Sales'],'salary':[50000,60000,70000,40000,45000,80000,90000,85000]})
df

,department,salary
0,HR,50000
1,HR,60000
2,HR,70000
3,IT,40000
4,IT,45000
5,IT,80000
6,Sales,90000
7,Sales,85000


In [92]:
sqldf("""
select department , avg(salary) as dep_avg_salary
from df
group by department
having avg(salary) > (select avg(salary_new)
                        from (select department , avg(salary) as salary_new
                                from df 
                                group by department))
""")

,department,dep_avg_salary
0,Sales,87500.0


## Q14

**Task:** Aggregation on Aggregation: Customer with maximum total spending.

In [93]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'amount':[500,700,300,1000,200,800]})
df

,order_id,customer_id,amount
0,1,101,500
1,2,101,700
2,3,102,300
3,4,103,1000
4,5,102,200
5,6,101,800


In [95]:
sqldf("""
SELECT CUSTOMER_ID , SUM(AMOUNT) AS MAX_AMMOUNT
FROM df
GROUP BY CUSTOMER_ID
HAVING SUM(AMOUNT) = (SELECT MAX(SUM_AMOUNT)
                        FROM (SELECT CUSTOMER_ID , SUM(AMOUNT) AS SUM_AMOUNT
                                FROM df
                                GROUP BY CUSTOMER_ID))


""")

,customer_id,MAX_AMMOUNT
0,101,2000


## Q15

**Task:** Mixed: Using CTE + DENSE_RANK(), find the 3rd highest product price in each category.

In [96]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6,7,8,9],'category':['A','A','A','B','B','B','C','C','C'],'price':[100,150,200,250,300,350,500,450,425]})
df

,product_id,category,price
0,1,A,100
1,2,A,150
2,3,A,200
3,4,B,250
4,5,B,300
5,6,B,350
6,7,C,500
7,8,C,450
8,9,C,425


In [99]:
sqldf("""
with my_cte as (
                select * 
                from (select *,
                Dense_rank()
                over(partition by category order by price desc) as rn
                from df) t
where rn = 3
)
select * from  my_cte

""")

,product_id,category,price,rn
0,1,A,100,3
1,4,B,250,3
2,9,C,425,3
